# MiniCode ? Passo 8: Treino Final QLoRA (Seeds 42/43/44 + Curva de Dados 25%/50%/100%)
- **8.a / 8.b:** Treina 3 adapters completos (`seed=42, 43, 44`) e reporta m?dia e desvio-padr?o da `val_loss`.
- **8.c:** Treina curva de escala (`25%`, `50%`, `100%` com `seed=42`).
- **8.d / 8.e:** Seleciona o melhor adapter exclusivamente pela `val_loss` (sem tocar no `test.jsonl`) e salva `training_metrics.json` + SHA-256.

In [ ]:
!nvidia-smi
import os, subprocess, sys
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'max_split_size_mb:128')
REPO_URL = 'https://github.com/jp-huppes/MiniCode.git'
REPO_DIR = Path('/content/MiniCode')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Commit ativo:', subprocess.check_output(['git', 'log', '-1', '--oneline'], text=True).strip())

In [ ]:
%pip install -q -e .
%pip install -q 'transformers>=4.46.0' 'peft>=0.13.0' 'trl>=0.12.0' 'bitsandbytes>=0.44.0' 'accelerate>=1.0.0' 'datasets>=3.0.0' matplotlib

In [ ]:
!PYTHONUTF8=1 python -m scripts.train_step8

In [ ]:
import json, shutil
from pathlib import Path
from IPython.display import Image, display

metrics = json.loads(Path('experiments/step8/training_metrics.json').read_text(encoding='utf-8'))
print('=== MULTI-SEED (100% TREINO) ===')
print(json.dumps({k: v for k, v in metrics['multi_seed_100pct'].items() if k != 'runs'}, indent=2))
print('\n=== ADAPTER SELECIONADO (8.d / 8.e) ===')
print(json.dumps({k: v for k, v in metrics['selected_adapter'].items() if k != 'files_sha256'}, indent=2))

display(Image(filename='experiments/step8/step8_curves.png'))

# Empacota m?tricas (para commit no GitHub) e adapters (para avalia??o local no Passo 9)
shutil.make_archive('/content/step8_experiments', 'zip', root_dir='.', base_dir='experiments/step8')
shutil.make_archive('/content/step8_checkpoints', 'zip', root_dir='.', base_dir='checkpoints/step8')
print('\nArquivos prontos para download:')
print('  1) /content/step8_experiments.zip (training_metrics.json + step8_curves.png)')
print('  2) /content/step8_checkpoints.zip (adapters LoRA + selected_adapter)')